In [ ]:
import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import json
from datetime import datetime
import glob

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def set_seed(seed):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class CodeT5PlusForPlagiarismDetection(nn.Module):
    def __init__(self, model_name="Salesforce/codet5p-220m"):
        """
        CodeT5+ model for plagiarism detection.
        Options for model_name:
        - "Salesforce/codet5p-220m" (smallest, fastest)
        - "Salesforce/codet5p-770m" (medium)
        - "Salesforce/codet5p-2b" (large, needs more GPU memory)
        """
        super().__init__()
        
        self.codet5plus = AutoModel.from_pretrained(model_name)
        
        if hasattr(self.codet5plus.config, 'd_model'):
            hidden_size = self.codet5plus.config.d_model
        elif hasattr(self.codet5plus.config, 'hidden_size'):
            hidden_size = self.codet5plus.config.hidden_size
        else:
            hidden_size = 768  
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.1),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, 2)  
        )
        
        for param in self.codet5plus.parameters():
            param.requires_grad = False
            
    def forward(self, input_ids, attention_mask, labels=None):
        if hasattr(self.codet5plus, 'encoder'):
            encoder_outputs = self.codet5plus.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True
            )
        else:
            encoder_outputs = self.codet5plus(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True
            )
        
        hidden_states = encoder_outputs.last_hidden_state  
        
        attention_mask_expanded = attention_mask.unsqueeze(-1).float()
        
        sum_hidden = torch.sum(hidden_states * attention_mask_expanded, dim=1)
        sum_mask = torch.sum(attention_mask_expanded, dim=1)
        sum_mask = torch.clamp(sum_mask, min=1e-9) 
        mean_pooled = sum_hidden / sum_mask
        

        masked_hidden = hidden_states.clone()
        masked_hidden[attention_mask_expanded == 0] = -1e9
        max_pooled, _ = torch.max(masked_hidden, dim=1)
        
        pooled_output = torch.cat([mean_pooled, max_pooled], dim=-1)
        
        if pooled_output.shape[-1] != mean_pooled.shape[-1] * 2:
            pooled_output = mean_pooled
        
        logits = self.classifier(pooled_output)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, 2), labels.view(-1))
            
        return loss, logits


class CodeT5PlusEnhanced(nn.Module):
    """CodeT5+ with attention pooling mechanism"""
    def __init__(self, model_name="Salesforce/codet5p-220m"):
        super().__init__()
        
        self.codet5plus = AutoModel.from_pretrained(model_name)
        
        if hasattr(self.codet5plus.config, 'd_model'):
            hidden_size = self.codet5plus.config.d_model
        else:
            hidden_size = 768
        
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Linear(hidden_size // 2, 2)
        )
        
        for param in self.codet5plus.parameters():
            param.requires_grad = False
            
    def forward(self, input_ids, attention_mask, labels=None):
        if hasattr(self.codet5plus, 'encoder'):
            encoder_outputs = self.codet5plus.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True
            )
        else:
            encoder_outputs = self.codet5plus(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True
            )
        
        hidden_states = encoder_outputs.last_hidden_state  
        
        attention_weights = self.attention(hidden_states)  
        attention_weights = attention_weights.squeeze(-1) 
        
        attention_weights = attention_weights.masked_fill(attention_mask == 0, -1e9)
        attention_weights = torch.softmax(attention_weights, dim=1)
        

        attention_weights = attention_weights.unsqueeze(-1)  
        pooled_output = torch.sum(hidden_states * attention_weights, dim=1) 
        logits = self.classifier(pooled_output)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, 2), labels.view(-1))
            
        return loss, logits


class CodeDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.codes = df["clean_code"].fillna("").astype(str).tolist()
        self.labels = df["label"].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.codes)
    
    def __getitem__(self, idx):
        code = self.codes[idx]
        label = self.labels[idx]
        
        inputs = self.tokenizer(
            code,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }


def train_codet5plus(model, train_loader, optimizer, criterion, device, epochs, seed, model_type="codet5plus"):
    """Train CodeT5+ model"""
    model.train()
    
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f"CodeT5+ - Epoch {epoch+1}/{epochs} (Seed {seed})")
        for batch in pbar:
            optimizer.zero_grad()
            
            loss, logits = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                labels=batch["labels"].to(device)
            )
            
            if loss is not None:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                total_loss += loss.item()
            
            _, predicted = torch.max(logits, 1)
            batch_total = batch["labels"].size(0)
            batch_correct = (predicted.cpu() == batch["labels"]).sum().item()
            
            total += batch_total
            correct += batch_correct
            
            avg_loss = total_loss / (pbar.n + 1) if total_loss > 0 else 0
            pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'acc': f'{100*batch_correct/batch_total:.2f}%'})
        
        epoch_acc = 100 * correct / total if total > 0 else 0
        avg_loss = total_loss / len(train_loader) if len(train_loader) > 0 else 0
        print(f"[Seed {seed}, CodeT5+] Epoch {epoch+1} Loss: {avg_loss:.4f}, Acc: {epoch_acc:.2f}%")
    
    return model


def evaluate_model(model, test_loader, device):
    """Evaluate model and return metrics"""
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Evaluating"):
            _, logits = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device)
            )
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch["labels"].cpu().numpy())
    
    if len(all_labels) > 0:
        accuracy = accuracy_score(all_labels, all_preds)
        precision = precision_score(all_labels, all_preds, average='binary', zero_division=0)
        recall = recall_score(all_labels, all_preds, average='binary', zero_division=0)
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    else:
        accuracy = precision = recall = f1 = 0.0
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'predictions': all_preds,
        'labels': all_labels
    }

def evaluate_all_test_files(model, tokenizer, device, seed, model_type="codet5plus"):
    """Evaluate model on all Test_0.csv to Test_9.csv files"""
    results = {}
    
    for i in range(10):
        test_file = f"Test_{i}.csv"
        if os.path.exists(test_file):
            try:
                df = pd.read_csv(test_file, usecols=["clean_code", "label"])
                df["clean_code"] = df["clean_code"].fillna("").astype(str)
                dataset = CodeDataset(df, tokenizer)
                loader = DataLoader(dataset, batch_size=8, shuffle=False)
                
                metrics = evaluate_model(model, loader, device)
                results[test_file] = metrics
                
                print(f"[Seed {seed}] {test_file}:")
                print(f"  Accuracy: {metrics['accuracy']:.4f}")
                print(f"  Precision: {metrics['precision']:.4f}")
                print(f"  Recall: {metrics['recall']:.4f}")
                print(f"  F1-Score: {metrics['f1']:.4f}")
                
            except Exception as e:
                print(f"Error evaluating {test_file}: {e}")
                results[test_file] = None
        else:
            print(f"File {test_file} not found.")
            results[test_file] = None
    
    valid_results = [v for v in results.values() if v is not None]
    if valid_results:
        avg_results = {
            'accuracy': np.mean([r['accuracy'] for r in valid_results]),
            'precision': np.mean([r['precision'] for r in valid_results]),
            'recall': np.mean([r['recall'] for r in valid_results]),
            'f1': np.mean([r['f1'] for r in valid_results]),
        }
        return avg_results, results
    else:
        return None, results


def run_codet5plus_experiment_with_seed(seed, train_file, epochs=5, model_size="220m", enhanced=False):
    """Run CodeT5+ experiment for a single seed"""
    print(f"\n{'='*80}")
    print(f"RUNNING CODET5+ EXPERIMENT WITH SEED: {seed}")
    print(f"{'='*80}")
    
    set_seed(seed)
    
    model_name = f"Salesforce/codet5p-{model_size}"
    print(f"Using model: {model_name}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    train_df = pd.read_csv(train_file, usecols=["clean_code", "label"])
    
    train_dataset = CodeDataset(train_df, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    
    if enhanced:
        model = CodeT5PlusEnhanced(model_name=model_name).to(device)
        model_type = "codet5plus_enhanced"
    else:
        model = CodeT5PlusForPlagiarismDetection(model_name=model_name).to(device)
        model_type = "codet5plus"
    
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    model = train_codet5plus(model, train_loader, optimizer, criterion, device, epochs, seed, model_type)
    
    print(f"\n[Seed {seed}] Evaluating CodeT5+ Model on test files...")
    codet5plus_avg_results, codet5plus_detailed_results = evaluate_all_test_files(model, tokenizer, device, seed, model_type)
    
    if codet5plus_avg_results:
        print(f"\n[Seed {seed}] CodeT5+ Average Results:")
        print(f"  Accuracy: {codet5plus_avg_results['accuracy']:.4f}")
        print(f"  Precision: {codet5plus_avg_results['precision']:.4f}")
        print(f"  Recall: {codet5plus_avg_results['recall']:.4f}")
        print(f"  F1-Score: {codet5plus_avg_results['f1']:.4f}")
    
    os.makedirs("models/codet5plus", exist_ok=True)
    model_path = f"models/codet5plus/codet5plus_{model_size}_{'enhanced_' if enhanced else ''}seed{seed}.pt"
    torch.save(model.state_dict(), model_path)
    print(f"[Seed {seed}] CodeT5+ model saved to: {model_path}")
    
    return {
        'seed': seed,
        'model_size': model_size,
        'enhanced': enhanced,
        'codet5plus_avg_results': codet5plus_avg_results,
        'codet5plus_detailed_results': codet5plus_detailed_results
    }


def load_codebert_results():
    """Load saved CodeBERT Stage 2 results"""
    codebert_results = {}
    
    if os.path.exists("results/aggregated_results.csv"):
        df = pd.read_csv("results/aggregated_results.csv")
        for _, row in df.iterrows():
            seed = int(row['seed'])
            codebert_results[seed] = {
                'accuracy': row['stage2_accuracy'],
                'precision': row['stage2_precision'],
                'recall': row['stage2_recall'],
                'f1': row['stage2_f1']
            }
        print(f"Loaded CodeBERT results for {len(codebert_results)} seeds from aggregated_results.csv")
        return codebert_results
    
    json_files = glob.glob("results/final_statistical_results_*.json")
    if json_files:
        latest_json = sorted(json_files)[-1]
        try:
            with open(latest_json, 'r') as f:
                stats_data = json.load(f)
            
            if 'summary' in stats_data:
                csv_files = glob.glob("results/*aggregated*.csv") + glob.glob("results/*final*.csv")
                if csv_files:
                    latest_csv = sorted(csv_files)[-1]
                    df = pd.read_csv(latest_csv)
                    for _, row in df.iterrows():
                        seed = int(row['seed'])
                        codebert_results[seed] = {
                            'accuracy': row.get('stage2_accuracy', row.get('accuracy', 0)),
                            'precision': row.get('stage2_precision', row.get('precision', 0)),
                            'recall': row.get('stage2_recall', row.get('recall', 0)),
                            'f1': row.get('stage2_f1', row.get('f1', 0))
                        }
                    print(f"Loaded CodeBERT results from {os.path.basename(latest_csv)}")
                    return codebert_results
        except Exception as e:
            print(f"Error loading JSON: {e}")
    
    interim_files = glob.glob("results/interim_results_*.csv")
    if interim_files:
        latest_interim = sorted(interim_files)[-1]
        try:
            df = pd.read_csv(latest_interim)
            df = df.drop_duplicates(subset=['seed'], keep='last')
            for _, row in df.iterrows():
                seed = int(row['seed'])
                codebert_results[seed] = {
                    'accuracy': row.get('stage2_accuracy', row.get('accuracy', 0)),
                    'precision': row.get('stage2_precision', row.get('precision', 0)),
                    'recall': row.get('stage2_recall', row.get('recall', 0)),
                    'f1': row.get('stage2_f1', row.get('f1', 0))
                }
            print(f"Loaded CodeBERT results from {os.path.basename(latest_interim)}")
            return codebert_results
        except Exception as e:
            print(f"Error loading interim file: {e}")
    
    print("\nNo automatic CodeBERT results found.")
    print("Options:")
    print("1. Enter path to CodeBERT results CSV")
    print("2. Continue without comparison")
    
    choice = input("\nEnter choice (1 or 2): ").strip()
    
    if choice == "1":
        results_path = input("Enter path to CodeBERT results CSV file: ").strip()
        if os.path.exists(results_path):
            try:
                df = pd.read_csv(results_path)
                for _, row in df.iterrows():
                    seed = int(row['seed'])
                    acc_col = None
                    for col in ['stage2_accuracy', 'accuracy', 'Accuracy', 'acc']:
                        if col in df.columns:
                            acc_col = col
                            break
                    
                    if acc_col:
                        codebert_results[seed] = {
                            'accuracy': row[acc_col],
                            'precision': row.get('stage2_precision', row.get('precision', row.get('Precision', 0))),
                            'recall': row.get('stage2_recall', row.get('recall', row.get('Recall', 0))),
                            'f1': row.get('stage2_f1', row.get('f1', row.get('F1', 0)))
                        }
                print(f"Loaded CodeBERT results for {len(codebert_results)} seeds")
            except Exception as e:
                print(f"Error loading file: {e}")
        else:
            print("File not found.")
    
    return codebert_results


def perform_comparison_analysis(codet5plus_results, codebert_results, metric='accuracy'):
    """Compare CodeT5+ vs CodeBERT results"""
    
    common_seeds = sorted(set(codet5plus_results.keys()) & set(codebert_results.keys()))
    
    if len(common_seeds) < 2:
        print(f"\nNot enough common seeds for comparison (need at least 2, got {len(common_seeds)})")
        print(f"CodeT5+ seeds: {sorted(codet5plus_results.keys())}")
        print(f"CodeBERT seeds: {sorted(codebert_results.keys())}")
        return None
    
    codet5plus_metrics = [codet5plus_results[seed][metric] for seed in common_seeds]
    codebert_metrics = [codebert_results[seed][metric] for seed in common_seeds]
    
    print(f"\n{'='*80}")
    print(f"COMPARISON ANALYSIS - {metric.upper()} METRIC")
    print(f"{'='*80}")
    print(f"Comparing {len(common_seeds)} common seeds: {common_seeds}")
    
    print(f"\nCodeT5+ {metric} across {len(common_seeds)} seeds:")
    print(f"  Mean: {np.mean(codet5plus_metrics):.4f}")
    print(f"  Std: {np.std(codet5plus_metrics):.4f}")
    print(f"  Min: {np.min(codet5plus_metrics):.4f}")
    print(f"  Max: {np.max(codet5plus_metrics):.4f}")
    
    print(f"\nCodeBERT Stage 2 {metric} across {len(common_seeds)} seeds:")
    print(f"  Mean: {np.mean(codebert_metrics):.4f}")
    print(f"  Std: {np.std(codebert_metrics):.4f}")
    print(f"  Min: {np.min(codebert_metrics):.4f}")
    print(f"  Max: {np.max(codebert_metrics):.4f}")
    
    differences = np.array(codet5plus_metrics) - np.array(codebert_metrics)
    print(f"\nDifferences (CodeT5+ - CodeBERT) in {metric}:")
    print(f"  Mean difference: {np.mean(differences):.4f}")
    print(f"  Std of difference: {np.std(differences):.4f}")
    print(f"  Min difference: {np.min(differences):.4f}")
    print(f"  Max difference: {np.max(differences):.4f}")
    print(f"  CodeT5+ better: {sum(i > 0 for i in differences)}/{len(differences)} seeds")
    print(f"  CodeBERT better: {sum(i < 0 for i in differences)}/{len(differences)} seeds")
    print(f"  Equal: {sum(i == 0 for i in differences)}/{len(differences)} seeds")
    
    t_stat, p_value = stats.ttest_rel(codet5plus_metrics, codebert_metrics)
    print(f"\nPaired t-test results (CodeT5+ vs CodeBERT):")
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value: {p_value:.6f}")
    
    try:
        w_stat, w_p_value = stats.wilcoxon(codet5plus_metrics, codebert_metrics)
        print(f"\nWilcoxon signed-rank test results:")
        print(f"  W-statistic: {w_stat:.4f}")
        print(f"  p-value: {w_p_value:.6f}")
    except:
        print(f"\nWilcoxon test could not be performed")
        w_stat, w_p_value = 0, 1.0
    
    if differences.std() > 0:
        d = np.mean(differences) / np.std(differences)
    else:
        d = 0
    print(f"\nEffect size (Cohen's d): {d:.4f}")
    
    print(f"\nStatistical Significance Interpretation:")
    if p_value < 0.05:
        if np.mean(differences) > 0:
            print(f"  ✅ CodeT5+ is SIGNIFICANTLY BETTER than CodeBERT (p < 0.05)")
        else:
            print(f"  ✅ CodeBERT is SIGNIFICANTLY BETTER than CodeT5+ (p < 0.05)")
        
        if abs(d) >= 0.8:
            print(f"  ✅ Large effect size (|d| = {abs(d):.2f})")
        elif abs(d) >= 0.5:
            print(f"  ⚠️  Medium effect size (|d| = {abs(d):.2f})")
        else:
            print(f"  ⚠️  Small effect size (|d| = {abs(d):.2f})")
    else:
        print(f"  ❌ No statistically significant difference (p = {p_value:.4f})")
    
    return {
        'metric': metric,
        'common_seeds': common_seeds,
        'codet5plus_mean': float(np.mean(codet5plus_metrics)),
        'codet5plus_std': float(np.std(codet5plus_metrics)),
        'codebert_mean': float(np.mean(codebert_metrics)),
        'codebert_std': float(np.std(codebert_metrics)),
        'difference_mean': float(np.mean(differences)),
        'difference_std': float(np.std(differences)),
        't_statistic': float(t_stat),
        'p_value': float(p_value),
        'w_statistic': float(w_stat),
        'wilcoxon_p': float(w_p_value),
        'cohens_d': float(d),
        'codet5plus_better': int(sum(i > 0 for i in differences)),
        'codebert_better': int(sum(i < 0 for i in differences)),
        'equal': int(sum(i == 0 for i in differences))
    }

def create_comparison_visualizations(codet5plus_metrics, codebert_metrics, differences, seeds_used, save_dir, metric_name="Accuracy", model_name="CodeT5+"):
    """Create comparison visualization plots"""
    seeds = seeds_used
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    axes[0, 0].plot(seeds, codet5plus_metrics, 'o-', label=model_name, linewidth=2, markersize=8, color='red')
    axes[0, 0].plot(seeds, codebert_metrics, 's-', label='CodeBERT Stage 2', linewidth=2, markersize=8, color='green')
    axes[0, 0].set_xlabel('Seed', fontsize=12)
    axes[0, 0].set_ylabel(metric_name, fontsize=12)
    axes[0, 0].set_title(f'{metric_name} Comparison Across Seeds', fontsize=14, fontweight='bold')
    axes[0, 0].legend(fontsize=12, loc='best')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].set_xticks(seeds)
    
    data_to_plot = [codet5plus_metrics, codebert_metrics]
    bp = axes[0, 1].boxplot(data_to_plot, labels=[model_name, 'CodeBERT\nStage 2'], patch_artist=True)
    
    colors = ['lightcoral', 'lightgreen']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    
    axes[0, 1].set_ylabel(metric_name, fontsize=12)
    axes[0, 1].set_title(f'Distribution Comparison', fontsize=14, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    for i, data in enumerate(data_to_plot):
        axes[0, 1].plot(i+1, np.mean(data), 'r_', markersize=15, markeredgewidth=2)
        axes[0, 1].text(i+1, np.mean(data) + 0.01, f'{np.mean(data):.3f}', 
                       ha='center', va='bottom', fontweight='bold')
    
    bars = axes[1, 0].bar(seeds, differences, 
                         color=['red' if x >= 0 else 'green' for x in differences], alpha=0.7)
    axes[1, 0].axhline(y=0, color='black', linestyle='-', alpha=0.5)
    axes[1, 0].set_xlabel('Seed', fontsize=12)
    axes[1, 0].set_ylabel(f'Difference ({model_name} - CodeBERT)', fontsize=12)
    axes[1, 0].set_title(f'Performance Difference Across Seeds', fontsize=14, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    axes[1, 0].set_xticks(seeds)
    
    for bar, diff in zip(bars, differences):
        height = bar.get_height()
        axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + (0.01 if height >= 0 else -0.02),
                       f'{diff:.4f}', ha='center', va='bottom' if height >= 0 else 'top', fontsize=9)
    
    avg_difference = np.mean(differences)
    axes[1, 0].axhline(y=avg_difference, color='blue', linestyle='--', 
                      linewidth=2, label=f'Average: {avg_difference:.4f}')
    axes[1, 0].legend()
    
    axes[1, 1].hist(differences, bins=min(10, len(differences)), edgecolor='black', alpha=0.7, color='skyblue')
    axes[1, 1].axvline(x=0, color='black', linestyle='-', linewidth=2, alpha=0.7)
    axes[1, 1].axvline(x=avg_difference, color='blue', linestyle='--', 
                      linewidth=2, label=f'Mean: {avg_difference:.4f}')
    axes[1, 1].set_xlabel(f'Difference ({model_name} - CodeBERT)', fontsize=12)
    axes[1, 1].set_ylabel('Frequency', fontsize=12)
    axes[1, 1].set_title(f'Distribution of Differences', fontsize=14, fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    fig.text(0.02, 0.02, 
             f'Summary: {model_name} Mean = {np.mean(codet5plus_metrics):.4f} ± {np.std(codet5plus_metrics):.4f}, '
             f'CodeBERT Mean = {np.mean(codebert_metrics):.4f} ± {np.std(codebert_metrics):.4f}, '
             f'Avg Difference = {avg_difference:.4f}',
             fontsize=10, style='italic', bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray", alpha=0.5))
    
    plt.tight_layout(rect=[0, 0.05, 1, 0.97])
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    fig_path_png = os.path.join(save_dir, f'comparison_{metric_name.lower()}_{timestamp}.png')
    fig_path_pdf = os.path.join(save_dir, f'comparison_{metric_name.lower()}_{timestamp}.pdf')
    
    plt.savefig(fig_path_png, dpi=300, bbox_inches='tight')
    plt.savefig(fig_path_pdf, bbox_inches='tight')
    plt.close()
    
    print(f"\nComparison visualizations saved to:")
    print(f"  {fig_path_png}")
    print(f"  {fig_path_pdf}")
    
    return fig_path_png


def main():
    """Main function to run CodeT5+ experiments and compare with CodeBERT"""
    print("="*80)
    print("CODET5+ VS CODEBERT - PLAGIARISM DETECTION COMPARISON")
    print("="*80)
    
    train_file = "Train.csv"
    
    if not os.path.exists(train_file):
        print(f"Error: Train file '{train_file}' not found!")
        print("Please ensure Train.csv exists in the current directory.")
        return
    
    print("\nLoading CodeBERT Stage 2 results...")
    codebert_results = load_codebert_results()
    
    if not codebert_results:
        print("Warning: No CodeBERT results found. Will only run CodeT5+ experiments.")
        compare_with_codebert = False
    else:
        compare_with_codebert = True
        print(f"Loaded CodeBERT results for seeds: {sorted(codebert_results.keys())}")
    
    print("\n" + "="*80)
    print("CODET5+ MODEL SELECTION")
    print("="*80)
    print("Available CodeT5+ models:")
    print("1. codet5p-220m (220 million parameters, fastest)")
    print("2. codet5p-770m (770 million parameters, balanced)")
    print("3. codet5p-2b (2 billion parameters, most accurate but needs more GPU memory)")
    
    model_choice = input("\nSelect model size (1, 2, or 3): ").strip()
    if model_choice == "1":
        model_size = "220m"
    elif model_choice == "2":
        model_size = "770m"
    elif model_choice == "3":
        model_size = "2b"
    else:
        print("Invalid choice. Using codet5p-220m (default).")
        model_size = "220m"
    
    print("\n" + "="*80)
    print("ARCHITECTURE SELECTION")
    print("="*80)
    print("Available architectures:")
    print("1. Basic CodeT5+ (mean+max pooling)")
    print("2. Enhanced CodeT5+ (attention pooling)")
    
    arch_choice = input("\nSelect architecture (1 or 2): ").strip()
    enhanced = (arch_choice == "2")
    
    DEFAULT_SEEDS = [42, 123, 456, 789, 999, 111, 222, 333, 444, 555]
    
    if compare_with_codebert:
        codebert_seeds = sorted(codebert_results.keys())
        print(f"\nCodeBERT was run on {len(codebert_seeds)} seeds: {codebert_seeds}")
        print(f"Default seeds available: {DEFAULT_SEEDS}")
        
        use_same_seeds = input(f"\nRun CodeT5+ on same seeds as CodeBERT? (y/n): ").strip().lower()
        
        if use_same_seeds == 'y':
            seeds_to_run = codebert_seeds
            print(f"Will run CodeT5+ on seeds: {seeds_to_run}")
        else:
            print(f"\nAvailable default seeds: {DEFAULT_SEEDS}")
            seeds_input = input(f"Enter seeds to run (comma-separated, e.g., '42,123,456'): ").strip()
            if seeds_input:
                try:
                    seeds_to_run = [int(s.strip()) for s in seeds_input.split(',')]
                except:
                    print("Invalid input. Using first 3 default seeds.")
                    seeds_to_run = DEFAULT_SEEDS[:3]
            else:
                seeds_to_run = DEFAULT_SEEDS[:3]
    else:
        print(f"\nDefault seeds available: {DEFAULT_SEEDS}")
        seeds_input = input(f"Enter seeds to run (comma-separated, press Enter for first 3): ").strip()
        if seeds_input:
            try:
                seeds_to_run = [int(s.strip()) for s in seeds_input.split(',')]
            except:
                print("Invalid input. Using first 3 default seeds.")
                seeds_to_run = DEFAULT_SEEDS[:3]
        else:
            seeds_to_run = DEFAULT_SEEDS[:3]
    
    try:
        epochs = int(input("\nNumber of training epochs for CodeT5+ (recommended: 3-5): ") or "3")
    except:
        epochs = 3
    
    print(f"\n{'='*80}")
    print("EXPERIMENT CONFIGURATION")
    print("="*80)
    print(f"  Model: CodeT5+ {model_size}")
    print(f"  Architecture: {'Enhanced (attention pooling)' if enhanced else 'Basic (mean+max pooling)'}")
    print(f"  Seeds: {seeds_to_run}")
    print(f"  Epochs: {epochs}")
    print(f"  Compare with CodeBERT: {compare_with_codebert}")
    
    confirm = input("\nProceed with CodeT5+ experiments? (y/n): ").strip().lower()
    if confirm != 'y':
        print("Experiment cancelled.")
        return
    
    model_dir = f"models/codet5plus_{model_size}{'_enhanced' if enhanced else ''}"
    results_dir = f"results/codet5plus_{model_size}{'_enhanced' if enhanced else ''}"
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)
    
    print(f"\n{'='*80}")
    print(f"STARTING CODET5+ EXPERIMENTS")
    print(f"{'='*80}")
    
    all_codet5plus_results = {}
    codet5plus_accuracies = []
    codet5plus_f1_scores = []
    
    start_time = datetime.now()
    
    for i, seed in enumerate(seeds_to_run, 1):
        print(f"\n[{i}/{len(seeds_to_run)}] ", end="")
        result = run_codet5plus_experiment_with_seed(seed, train_file, epochs, model_size, enhanced)
        
        if result['codet5plus_avg_results']:
            all_codet5plus_results[seed] = result['codet5plus_avg_results']
            codet5plus_accuracies.append(result['codet5plus_avg_results']['accuracy'])
            codet5plus_f1_scores.append(result['codet5plus_avg_results']['f1'])
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        interim_results = []
        for s, res in all_codet5plus_results.items():
            interim_results.append({
                'seed': s,
                'model_size': model_size,
                'enhanced': enhanced,
                'accuracy': res['accuracy'],
                'precision': res['precision'],
                'recall': res['recall'],
                'f1': res['f1']
            })
        
        if interim_results:
            interim_df = pd.DataFrame(interim_results)
            interim_path = f"{results_dir}/codet5plus_interim_results_{timestamp}.csv"
            interim_df.to_csv(interim_path, index=False)
            print(f"[Progress] CodeT5+ interim results saved to: {interim_path}")
    
    end_time = datetime.now()
    total_time = (end_time - start_time).total_seconds() / 60
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    if all_codet5plus_results:
        final_results = []
        for seed, res in all_codet5plus_results.items():
            final_results.append({
                'seed': seed,
                'model_size': model_size,
                'enhanced': enhanced,
                'accuracy': res['accuracy'],
                'precision': res['precision'],
                'recall': res['recall'],
                'f1': res['f1']
            })
        
        results_df = pd.DataFrame(final_results)
        results_path = f"{results_dir}/codet5plus_final_results_{timestamp}.csv"
        results_df.to_csv(results_path, index=False)
        
        print(f"\n{'='*80}")
        print("CODET5+ RESULTS SUMMARY")
        print(f"{'='*80}")
        print(f"Experiments completed in {total_time:.2f} minutes")
        print(f"Number of seeds: {len(seeds_to_run)}")
        print(f"Model: CodeT5+ {model_size}")
        print(f"Architecture: {'Enhanced (attention pooling)' if enhanced else 'Basic (mean+max pooling)'}")
        
        if codet5plus_accuracies:
            print(f"\nCodeT5+ Average Accuracy:")
            print(f"  Mean: {np.mean(codet5plus_accuracies):.4f} ± {np.std(codet5plus_accuracies):.4f}")
            print(f"  Range: [{np.min(codet5plus_accuracies):.4f}, {np.max(codet5plus_accuracies):.4f}]")
        
        if codet5plus_f1_scores:
            print(f"\nCodeT5+ Average F1-Score:")
            print(f"  Mean: {np.mean(codet5plus_f1_scores):.4f} ± {np.std(codet5plus_f1_scores):.4f}")
            print(f"  Range: [{np.min(codet5plus_f1_scores):.4f}, {np.max(codet5plus_f1_scores):.4f}]")
        
        print(f"\n📊 CodeT5+ results saved to: {results_path}")
        
        print(f"\n{'='*80}")
        print("CODET5+ PER-SEED RESULTS")
        print(f"{'='*80}")
        print(f"{'Seed':<8} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
        print(f"{'-'*60}")
        
        for seed in seeds_to_run:
            if seed in all_codet5plus_results:
                res = all_codet5plus_results[seed]
                print(f"{seed:<8} {res['accuracy']:<12.4f} {res['precision']:<12.4f} "
                      f"{res['recall']:<12.4f} {res['f1']:<12.4f}")
    
    if compare_with_codebert and all_codet5plus_results:
        print(f"\n{'='*80}")
        print("COMPARISON WITH CODEBERT STAGE 2")
        print(f"{'='*80}")
        
        comparison_acc = perform_comparison_analysis(all_codet5plus_results, codebert_results, metric='accuracy')
        
        comparison_f1 = perform_comparison_analysis(all_codet5plus_results, codebert_results, metric='f1')
        
        if comparison_acc and comparison_f1:
            common_seeds = comparison_acc['common_seeds']
            
            codet5plus_acc_common = [all_codet5plus_results[seed]['accuracy'] for seed in common_seeds]
            codebert_acc_common = [codebert_results[seed]['accuracy'] for seed in common_seeds]
            differences_acc = np.array(codet5plus_acc_common) - np.array(codebert_acc_common)
            
            codet5plus_f1_common = [all_codet5plus_results[seed]['f1'] for seed in common_seeds]
            codebert_f1_common = [codebert_results[seed]['f1'] for seed in common_seeds]
            differences_f1 = np.array(codet5plus_f1_common) - np.array(codebert_f1_common)
            
            model_name = f"CodeT5+ {model_size}{' Enhanced' if enhanced else ''}"
            
            fig_path_acc = create_comparison_visualizations(
                codet5plus_acc_common, codebert_acc_common, differences_acc, 
                common_seeds, results_dir, metric_name="Accuracy", model_name=model_name
            )
            
            fig_path_f1 = create_comparison_visualizations(
                codet5plus_f1_common, codebert_f1_common, differences_f1,
                common_seeds, results_dir, metric_name="F1-Score", model_name=model_name
            )
            
            comparison_stats = {
                'comparison_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                'model_name': model_name,
                'common_seeds': common_seeds,
                'accuracy_comparison': comparison_acc,
                'f1_comparison': comparison_f1,
                'summary': {
                    'codet5plus_accuracy_mean': float(np.mean(codet5plus_acc_common)),
                    'codet5plus_accuracy_std': float(np.std(codet5plus_acc_common)),
                    'codebert_accuracy_mean': float(np.mean(codebert_acc_common)),
                    'codebert_accuracy_std': float(np.std(codebert_acc_common)),
                    'codet5plus_f1_mean': float(np.mean(codet5plus_f1_common)),
                    'codet5plus_f1_std': float(np.std(codet5plus_f1_common)),
                    'codebert_f1_mean': float(np.mean(codebert_f1_common)),
                    'codebert_f1_std': float(np.std(codebert_f1_common))
                }
            }
            
            stats_path = f"{results_dir}/comparison_stats_{timestamp}.json"
            with open(stats_path, 'w') as f:
                json.dump(comparison_stats, f, indent=4)
            
            print(f"\n📈 Comparison statistics saved to: {stats_path}")
            print(f"📸 Comparison visualizations saved in '{results_dir}' directory")
            
            print(f"\n{'='*80}")
            print("COMPARISON SUMMARY TABLE")
            print(f"{'='*80}")
            print(f"{'Seed':<8} {'CodeT5+ Acc':<12} {'CodeBERT Acc':<12} {'Δ Acc':<10} {'CodeT5+ F1':<12} {'CodeBERT F1':<12} {'Δ F1':<10}")
            print(f"{'-'*80}")
            
            for seed in common_seeds:
                codet5plus_acc = all_codet5plus_results[seed]['accuracy']
                codebert_acc = codebert_results[seed]['accuracy']
                diff_acc = codet5plus_acc - codebert_acc
                
                codet5plus_f1 = all_codet5plus_results[seed]['f1']
                codebert_f1 = codebert_results[seed]['f1']
                diff_f1 = codet5plus_f1 - codebert_f1
                
                print(f"{seed:<8} {codet5plus_acc:<12.4f} {codebert_acc:<12.4f} {diff_acc:<10.4f} "
                      f"{codet5plus_f1:<12.4f} {codebert_f1:<12.4f} {diff_f1:<10.4f}")
    
    print(f"\n{'='*80}")
    print("EXPERIMENT COMPLETE")
    print(f"{'='*80}")
    print(f"Total time: {total_time:.2f} minutes")
    print(f"CodeT5+ models saved in: {model_dir}/")
    print(f"CodeT5+ results saved in: {results_dir}/")
    
    if compare_with_codebert and 'comparison_acc' in locals() and comparison_acc:
        print(f"\nComparison conclusion:")
        if comparison_acc['p_value'] < 0.05:
            if comparison_acc['difference_mean'] > 0:
                print(f"  ✅ CodeT5+ is SIGNIFICANTLY BETTER than CodeBERT for Accuracy")
            else:
                print(f"  ✅ CodeBERT is SIGNIFICANTLY BETTER than CodeT5+ for Accuracy")
        else:
            print(f"  ⚠️  No significant difference between CodeT5+ and CodeBERT for Accuracy")

if __name__ == "__main__":
    main()